# Design Spec — Per-Cell `code_type` & Language-Based Kernel Routing

**Status:** Approved (brainstorming) · **Date:** 2026-06-01 · **Graph snapshot:** `e5589c1f…`

## Summary

Today a SPUR notebook binds **one kernel per notebook slot** — every cell runs in the same language. This spec adds a per-cell `code_type` (`python` / `javascript` / `rust`) and makes the reactive engine **route each cell to its language's kernel**, so a single `run_cascade` can flow Python → Deno → (future) Rust. The Arrow **port store remains the polyglot boundary**: it is keyed by notebook identity alone, so any kernel — or any external process — reads and writes the same ports.

## Goals

- First-class, extensible `code_type` on code cells; language-based routing to the right kernel.
- A single cascade transparently spans languages; ports bridge them.
- Strict superset of today's behavior — existing single-language notebooks are untouched.

## Non-goals (this spec)

- **Rust execution.** `rust` is *registered but unprovisioned* — it routes, but errors cleanly until an `evcxr` kernelspec + a Rust `spur` port bootstrap are built (separate spec).
- **GUI cell-language picker.** MCP/backend only; TS bindings regenerate so the data is visible, but no toolbar dropdown this round (separate spec).
- **Parallel cross-language branch execution.** Cascade stays sequential; concurrency is a later optimization.

## 1 · Core model — two separated concerns

The whole design rests on splitting one overloaded string (`notebook:<path>`) into two **independent** identities:

| Concern | Keyed by | Decides | Today | After |
|---|---|---|---|---|
| **Port identity** | notebook **only** | where Arrow ports live | `f(path)` — *contaminated by slot id* | `f(path)` — **invariant, language-blind** |
| **Routing identity** | notebook **+** language | which kernel runs a cell | single `notebook:<path>` | `notebook:<path>#<spec>` per language |

> **Invariant (load-bearing):** the port store is keyed by notebook identity alone — never by kernel, language, or process. Routing decides *which kernel executes a cell*; it has **zero** influence on *where ports live*. The `#<spec>` suffix is a **dispatch-only** encoding, stripped before any port-root derivation.

### `code_type` registry

A cell carries `code_type` — the **language label**, not the kernelspec. A single registry maps language → kernelspec → bootstrap, consulted by routing, pre-flight boot, and cell wrapping alike:

| `code_type` | kernelspec (`spec_name`) | port bootstrap | status |
|---|---|---|---|
| `python` | `python3` | `python_bootstrap` | ✅ works today |
| `javascript` | `deno` | `javascript_bootstrap` | ✅ works today |
| `rust` | `evcxr` | *(none yet)* | ⛔ registered → clean error |

Storing the **language** (not the spec) keeps cell metadata stable if the runtime behind a language changes (e.g. swap Deno for Node); the registry absorbs it.

## 2 · System architecture

Two Rust layers separated by a **dynamic JSON-RPC bridge** (the AgentBridge). The engine resolves language and builds the dispatch slot id; the tauri layer wraps the cell with the right port bootstrap and dispatches to the matching kernel. Every kernel reads/writes **one** notebook-keyed port store.

```mermaid
flowchart TB
  subgraph Engine["spur-notebook · reactive engine (Rust)"]
    RC["run_cascade / run_cell"]
    PF["pre-flight ensure kernels"]
    RES["resolve code_type<br/>explicit → kernelspec → python"]
    CRR["cell_run_request<br/>slot_id = notebook:&lt;path&gt;#&lt;spec&gt;"]
    RC --> PF --> RES --> CRR
  end

  subgraph Bridge["AgentBridge · JSON-RPC — dynamic boundary (no static edge)"]
    P["run_cell params<br/>{ cell_id, kernel_id, notebook_path, code, expected_version }"]
  end

  subgraph Tauri["jute-notebook/src-tauri · kernel layer (Rust)"]
    RCE["run_cell_events"]
    WCK["wrap_cell_for_kernel(notebook_path, spec, code)"]
    BOOT["python_bootstrap | javascript_bootstrap"]
    RCE --> WCK --> BOOT
  end

  subgraph Kernels["kernel slots · state.kernels (multi-slot map)"]
    K1(["notebook:&lt;path&gt;#python3"])
    K2(["notebook:&lt;path&gt;#deno"])
    K3(["notebook:&lt;path&gt;#evcxr · future"]):::future
  end

  subgraph PortsBox["~/.spur/notebooks/&lt;id&gt;/ports — keyed by NOTEBOOK ONLY"]
    M[("manifest.json + *.arrow")]
  end

  CRR -- "kernel_id + notebook_path" --> P --> RCE
  BOOT -- dispatch --> K1 & K2 & K3
  K1 & K2 & K3 -- "spur.put / spur.get (Arrow IPC)" --> M

  classDef future stroke-dasharray:5 5,opacity:.6;
```

The bridge crossing is the boundary the static call graph **cannot see** — both the MCP `run_cell` tool and the tauri command change together and are pinned by a contract test (§7).

## 3 · `code_type` resolution & the mandatory/legacy rule

`code_type` is **required** on every code cell created or edited through the API, and **optional only in the on-disk schema** — its absence marks a *legacy* cell that resolves via the fallback chain.

```mermaid
flowchart TD
  A["cell about to run"] --> B{"cell.spur.code_type present?"}
  B -- "yes (all new cells)" --> C["use explicit code_type"]
  B -- "no — legacy on disk" --> D{"notebook.kernelspec set?"}
  D -- yes --> E["map kernelspec → code_type"]
  D -- no --> F["default: python"]
  C --> G["kernelspec_for(code_type)"]
  E --> G
  F --> G
  G --> H["slot_id = notebook:&lt;path&gt;#&lt;spec&gt;"]

  style C fill:#14532d,color:#fff
  style E fill:#713f12,color:#fff
  style F fill:#7f1d1d,color:#fff
```

| Moment | Rule |
|---|---|
| **Create / edit (API)** | `kind=code` ⇒ `code_type` **required** — reject if missing. `kind∈{markdown,raw}` ⇒ `code_type` must be absent. |
| **Read (legacy on disk)** | untyped code cell ⇒ resolve **explicit → notebook kernelspec → `python`**. No error; **resolve-on-read, bytes untouched** (the field gets stamped only when the cell is next edited). |

This confines ambiguity to pre-existing data; every new cell is unambiguous about its language.

## 4 · Metadata shape & MCP tool surface

### Metadata (`backend/notebook.rs`)

```rust
#[derive(Clone, Debug, Serialize, Deserialize, PartialEq, Eq, TS)]
#[serde(rename_all = "lowercase")]
pub enum CodeType { Python, Javascript, Rust }

pub struct SpurCellMetadata {
    pub version: u64,
    pub last_edited_by: Option<String>,
    pub datasource_setup: Option<bool>,
    pub dag: Option<CellDagMetadata>,
    #[serde(skip_serializing_if = "Option::is_none")]
    pub code_type: Option<CodeType>,   // Option = on-disk/legacy only; API enforces presence
}
```

- `Option<CodeType>` keeps the **on-disk schema additive** (legacy cells omit it); the **API layer** enforces presence on code cells.
- Lives directly under `spur`, **beside `dag` — not inside it** (language is orthogonal to produces/consumes wiring).
- `#[ts(export)]` regenerates `SpurCellMetadata.ts` + a new `CodeType.ts` so the field is visible to the frontend (no picker yet).

### MCP tools (both additive)

1. **`notebook_insert_cell`** gains a `code_type` param. Validation: `kind=code` ⇒ required; `kind∈{markdown,raw}` ⇒ forbidden.
2. **`notebook_set_cell_code_type { id, code_type, expected_version }`** — new tool, a direct mirror of `notebook_set_dag_metadata` (same optimistic-concurrency `expected_version` protocol, same `spur` facet merge). Also the path that stamps `code_type` onto a legacy cell on first edit.

## 5 · Routing change & cascade sequence

The change is localized to **`cell_run_request`** (`engine.rs:461` — the single-caller seam). `ReactiveEngine`'s `kernel_id: String` becomes a per-cell resolution; the notebook path is retained as the slot-id base and port-root anchor.

```rust
fn slot_id_for(notebook_path: &str, code_type: CodeType) -> String {
    let spec = kernelspec_for(code_type);                     // python→python3, javascript→deno, rust→evcxr
    format!("{}#{}", notebook_slot_id(notebook_path), spec)    // "notebook:<path>#python3"
}
// cell_run_request: resolve cell.code_type (§3) → kernel_id = Some(slot_id_for(path, ct))
```

Everything downstream already routes on `request.kernel_id` (the runner passes it straight through the bridge; `state.kernels` is a map), so cascade ordering, the `stale_version` protocol, and the runner are **unchanged**.

```mermaid
sequenceDiagram
  autonumber
  participant U as caller (MCP)
  participant E as ReactiveEngine
  participant S as state.kernels
  participant KP as python3 slot
  participant KD as deno slot
  participant PT as ports (notebook-keyed)

  U->>E: run_cascade(source)
  E->>E: scan DAG → distinct specs {python3, deno}
  par pre-flight ensure (parallel)
    E->>S: ensure notebook:path#python3
    S-->>KP: provision + start
  and
    E->>S: ensure notebook:path#deno
    S-->>KD: provision + start
  end
  loop each cell in topological order
    E->>E: resolve code_type → slot_id
    alt python cell
      E->>KP: run (wrap_python_cell, port_root=f(path))
      KP->>PT: spur.put(port)
    else javascript cell
      E->>KD: run (wrap_js_cell, port_root=f(path))
      PT-->>KD: spur.get(port)
      KD->>PT: spur.put(port)
    end
  end
```

## 6 · The port-root invariant (decoupling — approach 4b)

**The flaw (found via graph analyst):** `port_root_for_kernel_id` (`commands.rs:1440`) derives the port directory from the slot-id string by stripping only the `notebook:` prefix. With a composite `notebook:<path>#deno`, that yields `<path>#deno` → a **forked, per-language port directory** → Python-writes / Deno-reads break silently.

**Fix (4b — structural, not a parse patch):** the engine already *knows* the notebook path. Thread it down explicitly; never reconstruct it from the routing string.

- `run_cell` bridge params gain `notebook_path`.
- `wrap_cell_for_kernel(notebook_path, spec, code)` → `notebook_port_root(notebook_path)` directly.
- `port_root_for_kernel_id` is **deleted**.

```mermaid
flowchart LR
  subgraph R["routing identity · per language"]
    SP["notebook:/nb#python3"]
    SD["notebook:/nb#deno"]
  end
  subgraph N["notebook identity · the ONLY port key"]
    NP["notebook_path = /nb"]
  end
  SP -- "dispatch only" --> KP["python3 kernel"]
  SD -- "dispatch only" --> KD["deno kernel"]
  SP -. "threaded explicitly, never parsed" .-> NP
  SD -. "threaded explicitly, never parsed" .-> NP
  NP --> PR[("~/.spur/notebooks/&lt;id&gt;/ports<br/>ONE shared root")]
  KP --> PR
  KD --> PR
```

**Executable form of the invariant (regression test):** the same notebook resolved under `python3` and under `deno` produces **byte-identical** port roots. After 4b the port root is computed once from notebook identity and can't be re-derived from a routing string — the invariant becomes impossible to violate by construction.

## 7 · Kernel lifecycle — pre-flight ensure

Cascades boot all needed kernels **up front, in parallel**; single-cell runs boot lazily. Pre-flight runs *before any dispatch*, so it never races the engine's existing debounce / in-flight slot accounting.

```mermaid
stateDiagram-v2
  [*] --> Scan: run_cascade
  Scan --> Ensure: distinct specs from DAG
  state Ensure {
    [*] --> Check
    Check --> Live: slot already in state.kernels
    Check --> Provision: missing
    Provision --> Live: ensure_kernelspec + start_kernel(slot_id)
    Provision --> FailFast: spec = evcxr (rust)
  }
  Live --> Dispatch: all required kernels up
  Dispatch --> Dispatch: cell → resolve → slot
  Dispatch --> [*]: cascade complete
  FailFast --> [*]: error names offending cells — nothing runs
```

- **Reuse, not new machinery:** `start_kernel` already accepts an explicit `slot_id` and provisions per spec; `provisioning_target_for_spec` (`start_kernel.rs:101`) just gains an `evcxr` arm that returns the *not-yet-supported* signal.
- **Rust = fail-fast, fail-clear:** detected in pre-flight *before running anything*; structured error names the offending cells. No partial cascade, no orphaned ports.
- **Single-cell `run_cell`:** lazy-ensure just that cell's slot, then dispatch (exactly how the demo's Deno cell ran — minus the manual `start_kernel`).

## 8 · Worked example — the polyglot diamond

The demo notebook, now as one cross-language cascade. Pre-flight boots `python3` + `deno`; the engine routes each node to its slot; the four Arrow ports live in the single notebook-keyed store.

```mermaid
flowchart TD
  R["raw_sales<br/><b>python</b> · source"] -- raw_sales --> E["enriched_sales<br/><b>python</b>"]
  E -- enriched_sales --> RG["region_summary<br/><b>python</b>"]
  E -- enriched_sales --> PR["product_summary<br/><b>python</b>"]
  RG -- region_summary --> REP["report<br/><b>python</b>"]
  PR -- product_summary --> REP
  RG -- region_summary --> VIZ["report_chart<br/><b>javascript · deno</b>"]
  PR -- product_summary --> VIZ

  classDef py fill:#1e3a5f,color:#fff,stroke:#38bdf8;
  classDef js fill:#4c1d95,color:#fff,stroke:#a78bfa;
  class R,E,RG,PR,REP py;
  class VIZ js;
```

A single `run_cascade(raw_sales)` now executes the Python nodes on `…#python3` and `report_chart` on `…#deno` — no kernel swap, no manual step. The Deno node reads `region_summary` / `product_summary` from the same ports the Python nodes wrote.

## 9 · Error contract

| Situation | Behavior |
|---|---|
| code cell with no `code_type` (create/edit) | reject — `invalid_params`, "code_type required for code cells" |
| `code_type` on markdown/raw | reject — "code_type only valid on code cells" |
| unknown `code_type` value | reject at deserialization (closed enum) |
| `code_type: rust` in a cascade | pre-flight **fail-fast**, names offending cells, runs nothing |
| `code_type: rust` single-cell run | lazy-boot error, same clear message |
| legacy untyped cell | **no error** — resolve-on-read → notebook kernelspec → `python` |

## 10 · Testing, blast radius & build sequence

### Tests (must-haves first)

- **Port-root invariant** — same notebook under `python3` vs `deno` ⇒ byte-identical port root. *(executable form of §6)*
- **Polyglot cascade** — Python source → Deno consumer; pre-flight boots both; Deno reads the Python-written port. *(extends `reactive_dag.rs`; base: `deno_write_port_is_readable_from_deno_and_rust`)*
- **Bridge-contract pin** — `run_cell` params (incl. `notebook_path`) match across MCP tool ⇄ tauri command. *(guards the dynamic boundary)*
- **Unit** — registry mapping, `slot_id_for` encoding, resolution chain (explicit→kernelspec→python).
- **Validation** — mandatory `code_type` on `insert_cell`; `set_cell_code_type` optimistic concurrency (mirror `set_dag_metadata`).
- **Legacy fallback** — untyped cell runs under notebook kernelspec.
- **Update existing** — `run_cell_chokepoint_wraps_*` (signature), `run_cell_request_uses_notebook_path_slot_id` (composite id), `reactive_dag` harness.

### Blast radius (from graph analyst — all low/contained)

| Symbol / area | callers | role in change |
|---|---|---|
| `cell_run_request` | 1 | routing seam — primary edit |
| `port_root_for_kernel_id` → deleted | 1 | port decoupling (4b) |
| `wrap_cell_for_kernel` | 4 (3 tests) | signature: `notebook_path` |
| `notebook_slot_id` | 11 | slot-id base — composite encoding |
| `SpurCellMetadata` / `CellDagMetadata` | 0 (serde) | additive field + TS regen |
| `provisioning_target_for_spec` | — | `evcxr` arm |

> ⚠️ **Churn hotspot:** every touched file was modified within ~48h of the snapshot (`engine.rs`, `mcp/mod.rs`, `commands.rs`, `backend/notebook.rs`). Land in a tight, well-sequenced PR; rebase aggressively.

### Build sequence

1. **Metadata + registry** — `CodeType`, `SpurCellMetadata.code_type`, `kernelspec_for`, TS bindings. *(no behavior change)*
2. **Port-root decoupling (4b)** — thread `notebook_path`; delete `port_root_for_kernel_id`; **invariant test goes green first**.
3. **Routing seam** — `slot_id_for` + `cell_run_request` resolution; update slot-id tests.
4. **Pre-flight lifecycle** — distinct-spec scan + parallel ensure; `evcxr` fail-fast.
5. **MCP surface** — `insert_cell` validation + `set_cell_code_type`.
6. **Polyglot integration test** — the §8 diamond, end to end.